In [2]:
%pip install ultralytics opencv-python

  Using cached ultralytics-8.4.47-py3-none-any.whl.metadata (39 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached torchvision-0.26.0-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached polars-1.40.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.40.1-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
Using cached ultralytics-8.4.47-py3-none-any.whl (1.2 MB)
Using cached polars-1.40.1-py3-none-any.whl (828 kB)
Using cached polars_runtime_32-1.40.1-cp310-abi3-win_amd64.whl (51.8 MB)
Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl (154 kB)
Using cached torchvision-0.26.0-cp313-cp313-win_amd64.whl (4.3 MB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   -----------------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import json
import os
from collections import Counter

DATASET_PATH = './tt100k_2021/tt100k_2021'
TARGET_CLASSES = ['pl120', 'pl40', 'pne', 'p5', 'pl100', 'pl5', 'pl80', 'p11', 'pl60', 'po', 'p10', 'pl30', 'p26', 'p6', 'pg', 'ph4', 'pm30', 'pm55', 'p12', 'pm20', 'pl20',
                  'pm10', 'pl70', 'pl10', 'p19', 'p3', 'pr40', 'pn', 'p13', 'p23', 'p27', 'p1', 'pb', 'p16', 'pnl', 'ph4.5', 'ph5', 'p2', 'pa14', 'pm2', 'p15', 'pm15', 'p17', 'p18', 'pm13']

with open(os.path.join(DATASET_PATH, 'annotations_all.json')) as f:
    data = json.load(f)

# Count every occurrence of classes in the raw data
all_found_categories = []
for img_id, info in data['imgs'].items():
    for obj in info['objects']:
        if obj['category'] in TARGET_CLASSES:
            all_found_categories.append(obj['category'])

stats = Counter(all_found_categories)
print(f"📊 Total Target Instances Found: {len(all_found_categories)}")
print(f"✅ Unique Classes Found: {len(stats)}/45")

missing = set(TARGET_CLASSES) - set(stats.keys())
if missing:
    print(f"❌ WARNING: These classes are missing from your JSON: {missing}")
else:
    print("🚀 All 45 classes exist in the raw dataset.")

📊 Total Target Instances Found: 17489
✅ Unique Classes Found: 43/45
❌ WARNING: These classes are missing from your JSON: {'po', 'pnl'}


In [17]:
import json
import os
import pandas as pd
from collections import Counter

# Based on your VS Code explorer: tt100k_2021 -> tt100k_2021 -> annotations_all.json
ANNOTATIONS_PATH = './tt100k_2021/tt100k_2021/annotations_all.json'
TARGET_CLASSES = ['pl120', 'pl40', 'pne', 'p5', 'pl100', 'pl5', 'pl80', 'p11', 'pl60', 'po', 'p10', 'pl30', 'p26', 'p6', 'pg', 'ph4', 'pm30', 'pm55', 'p12', 'pm20', 'pl20',
                  'pm10', 'pl70', 'pl10', 'p19', 'p3', 'pr40', 'pn', 'p13', 'p23', 'p27', 'p1', 'pb', 'p16', 'pnl', 'ph4.5', 'ph5', 'p2', 'pa14', 'pm2', 'p15', 'pm15', 'p17', 'p18', 'pm13']

if os.path.exists(ANNOTATIONS_PATH):
    with open(ANNOTATIONS_PATH, 'r') as f:
        data = json.load(f)

    # Count occurrences in the raw original images
    raw_counts = Counter()
    for img_id, info in data['imgs'].items():
        for obj in info['objects']:
            cat = obj['category']
            if cat in TARGET_CLASSES:
                raw_counts[cat] += 1

    # Format the data for a clean report
    dist_data = []
    for cls in TARGET_CLASSES:
        count = raw_counts[cls]
        dist_data.append({
            "Class Name": cls,
            "Total Instances": count,
            "Status": "✅ Found" if count > 0 else "❌ Missing from JSON"
        })

    df_original = pd.DataFrame(dist_data).sort_values(
        by="Total Instances", ascending=False)

    print("📊 ORIGINAL TT100K_2021 RAW DISTRIBUTION")
    print("="*50)
    print(df_original.to_string(index=False))
    print("="*50)


📊 ORIGINAL TT100K_2021 RAW DISTRIBUTION
Class Name  Total Instances              Status
        pn             3176             ✅ Found
       pne             2384             ✅ Found
       p11             1582             ✅ Found
      pl40             1413             ✅ Found
      pl80              904             ✅ Found
       p26              840             ✅ Found
      pl60              835             ✅ Found
     pl100              673             ✅ Found
      pl30              640             ✅ Found
       pl5              537             ✅ Found
        p5              421             ✅ Found
       p13              379             ✅ Found
       p10              374             ✅ Found
     pl120              298             ✅ Found
       p23              297             ✅ Found
      pr40              201             ✅ Found
       p12              189             ✅ Found
     ph4.5              186             ✅ Found
        p3              173             ✅ Found


In [18]:
# The 40 most frequent classes based on your audit
TARGET_CLASSES = [
    'pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5',
    'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20',
    'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30',
    'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2'
]

print(f"🎯 Final Target Class Count: {len(TARGET_CLASSES)}")

🎯 Final Target Class Count: 40


In [19]:
from sklearn.model_selection import train_test_split

# Map images to their primary class for stratification
img_to_main_class = {}
for img_id, info in data['imgs'].items():
    classes_in_img = [obj['category']
                      for obj in info['objects'] if obj['category'] in TARGET_CLASSES]
    if classes_in_img:
        # Assign image to the rarest class it contains to protect minority classes
        main_cls = min(classes_in_img, key=lambda x: raw_counts[x])
        img_to_main_class[img_id] = main_cls

ids = list(img_to_main_class.keys())
labels = [img_to_main_class[i] for i in ids]

# Phase 1: Split 70% into Train, 30% into Temp (Val + Test)
train_ids, temp_ids = train_test_split(
    ids, test_size=0.30, random_state=42, stratify=labels)

# Phase 2: Split Temp 50/50 to get exactly 15% Val and 15% Test
temp_labels = [img_to_main_class[i] for i in temp_ids]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, random_state=42, stratify=temp_labels)

print(f"📈 Distribution Successful:")
print(f"   - Train: {len(train_ids)} images")
print(f"   - Val  : {len(val_ids)} images")
print(f"   - Test : {len(test_ids)} images")

📈 Distribution Successful:
   - Train: 6162 images
   - Val  : 1320 images
   - Test : 1321 images


In [20]:
import cv2
import numpy as np
from tqdm import tqdm

OUTPUT_ROOT = './yolo_dataset'
TILE_SIZE = 640
OVERLAP = 0.2


def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)


def process_and_tile(ids, split_name):
    os.makedirs(f"{OUTPUT_ROOT}/{split_name}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_ROOT}/{split_name}/labels", exist_ok=True)

    print(f"🛠️ Processing {split_name}...")
    for img_id in tqdm(ids):
        img_info = data['imgs'][img_id]
        img = cv2.imread(os.path.join(DATASET_PATH, img_info['path']))
        if img is None:
            continue

        img = apply_clahe(img)
        h, w, _ = img.shape
        step = int(TILE_SIZE * (1 - OVERLAP))

        for y in range(0, h - TILE_SIZE + 1, step):
            for x in range(0, w - TILE_SIZE + 1, step):
                tile_labels = []
                for obj in img_info['objects']:
                    if obj['category'] not in TARGET_CLASSES:
                        continue
                    bx = obj['bbox']
                    cx, cy = (bx['xmin'] + bx['xmax']) / \
                        2, (bx['ymin'] + bx['ymax'])/2
                    if x <= cx < x+TILE_SIZE and y <= cy < y+TILE_SIZE:
                        tx, ty = (cx - x)/TILE_SIZE, (cy - y)/TILE_SIZE
                        tw, th = (bx['xmax']-bx['xmin']) / \
                            TILE_SIZE, (bx['ymax']-bx['ymin'])/TILE_SIZE
                        tile_labels.append(
                            f"{TARGET_CLASSES.index(obj['category'])} {tx:.6f} {ty:.6f} {tw:.6f} {th:.6f}")

                if tile_labels:
                    name = f"{img_id}_{y}_{x}"
                    cv2.imwrite(
                        f"{OUTPUT_ROOT}/{split_name}/images/{name}.jpg", img[y:y+TILE_SIZE, x:x+TILE_SIZE])
                    with open(f"{OUTPUT_ROOT}/{split_name}/labels/{name}.txt", 'w') as f:
                        f.write("\n".join(tile_labels))


process_and_tile(train_ids, 'train')
process_and_tile(val_ids, 'val')
process_and_tile(test_ids, 'test')

🛠️ Processing train...


100%|██████████| 6162/6162 [10:30<00:00,  9.78it/s]


🛠️ Processing val...


100%|██████████| 1320/1320 [02:18<00:00,  9.55it/s]


🛠️ Processing test...


100%|██████████| 1321/1321 [02:12<00:00,  9.95it/s]


In [23]:
import os
import yaml

ORIGINAL_NAMES = [
    'pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5',
    'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20',
    'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30',
    'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2'
]

# 2. Create the Clean 40-Class YAML
data_config = {
    'train': os.path.join(OUTPUT_ROOT, 'train'),
    'val': os.path.join(OUTPUT_ROOT, 'val'),
    'test': os.path.join(OUTPUT_ROOT    , 'test'),
    'nc': 40,  # Updated to 40
    'names': ORIGINAL_NAMES
}

# Delete old and write new
if os.path.exists('./yolo_dataset/data.yaml'):
    os.remove('./yolo_dataset/data.yaml')

with open('./yolo_dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(
    f"✅ YAML Updated for SPICSCON: {len(ORIGINAL_NAMES)} classes registered.")

✅ YAML Updated for SPICSCON: 40 classes registered.


In [24]:
import os

# Use the INPUT_DIR path identified earlier
OUTPUT_ROOT = './yolo_dataset'


def summarize_split(split_name):
    img_dir = os.path.join(OUTPUT_ROOT, split_name, 'images')
    lbl_dir = os.path.join(OUTPUT_ROOT,  split_name, 'labels')

    if not os.path.exists(img_dir):
        return None

    images = [f for f in os.listdir(
        img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    labels = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]

    # Count unique classes in this split
    unique_classes = set()
    for lbl in labels:
        with open(os.path.join(lbl_dir, lbl), 'r') as f:
            for line in f:
                parts = line.split()
                if parts:
                    unique_classes.add(parts[0])

    return {
        "images": len(images),
        "labels": len(labels),
        "classes": len(unique_classes)
    }


print("📊 YOLO_DATASET AUDIT REPORT")
print("="*40)

for split in ['train', 'val', 'test']:
    stats = summarize_split(split)
    if stats:
        print(f"📁 {split.upper()} SPLIT:")
        print(f"   - Total Images  : {stats['images']}")
        print(f"   - Total Labels  : {stats['labels']}")
        print(f"   - Unique Classes: {stats['classes']}")
    else:
        print(f"⚠️  {split.upper()} split not found.")
print("="*40)

📊 YOLO_DATASET AUDIT REPORT
📁 TRAIN SPLIT:
   - Total Images  : 8144
   - Total Labels  : 8144
   - Unique Classes: 40
📁 VAL SPLIT:
   - Total Images  : 1707
   - Total Labels  : 1707
   - Unique Classes: 40
📁 TEST SPLIT:
   - Total Images  : 1716
   - Total Labels  : 1716
   - Unique Classes: 40


In [25]:
import os

OUTPUT_ROOT = './yolo_dataset'

# Collect unique classes across all splits
all_unique_classes = set()
for split in ['train', 'val', 'test']:
    lbl_dir = os.path.join(OUTPUT_ROOT, split, 'labels')
    if os.path.exists(lbl_dir):
        labels = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
        for lbl in labels:
            with open(os.path.join(lbl_dir, lbl), 'r') as f:
                for line in f:
                    parts = line.split()
                    if parts:
                        all_unique_classes.add(parts[0])

print("Unique classes across all splits:", sorted(all_unique_classes))

Unique classes across all splits: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '5', '6', '7', '8', '9']


In [26]:
import os

# Your final 40-class list used for the stratified split
TARGET_CLASSES = [
    'pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5',
    'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20',
    'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30',
    'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2'
]


def get_unique_names(split):
    lbl_dir = os.path.join('./yolo_dataset', split, 'labels')
    found_ids = set()

    if os.path.exists(lbl_dir):
        for lbl in os.listdir(lbl_dir):
            with open(os.path.join(lbl_dir, lbl), 'r') as f:
                for line in f:
                    parts = line.split()
                    if parts:
                        found_ids.add(int(parts[0]))

    # Map IDs back to names
    return [TARGET_CLASSES[i] for i in sorted(list(found_ids))]


# Check your main splits
train_names = get_unique_names('train')
val_names = get_unique_names('val')
test_names = get_unique_names('test')

print(f"✅ Found {len(train_names)} unique classes in TRAIN:")
print(train_names)

print(f"\n✅ Found {len(val_names)} unique classes in VAL:")
print(val_names)

print(f"\n✅ Found {len(test_names)} unique classes in TEST:")
print(test_names)

✅ Found 40 unique classes in TRAIN:
['pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5', 'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20', 'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30', 'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2']

✅ Found 40 unique classes in VAL:
['pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5', 'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20', 'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30', 'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2']

✅ Found 40 unique classes in TEST:
['pn', 'pne', 'p11', 'pl40', 'pl80', 'p26', 'pl60', 'pl100', 'pl30', 'pl5', 'p5', 'p13', 'p10', 'pl120', 'p23', 'pr40', 'p12', 'ph4.5', 'p3', 'pl20', 'pg', 'pm20', 'pl70', 'pm55', 'p27', 'p19', 'ph4', 'ph5', 'p6', 'pm30', 'pb', 'p18', 'p1', 'pa14', 'pm10', 'pl10', 'p17', 'pm15', 'p16', 'p2']
